# Detecção de Fraudes em Cartão de Crédito\n**Desafio de Projeto — DIO | Bradesco — Dados, Cibersegurança & GenAI**\n\nComparação de modelos com foco no **Recall da fraude**, ajuste do limiar de decisão e explicabilidade com SHAP.

In [ ]:
import warnings\nwarnings.filterwarnings('ignore')\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, average_precision_score\nfrom xgboost import XGBClassifier\nimport shap\nRANDOM_STATE = 42

## 1. Carregamento e exploração

In [ ]:
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'\ntry:\n    df = pd.read_csv(DATA_URL)\nexcept Exception:\n    df = pd.read_csv('creditcard.csv')\nprint('Dimensão:', df.shape)\ndisplay(df.head())\nprint('Valores ausentes:', int(df.isnull().sum().sum()))\nprint(df['Class'].value_counts())\nprint(f\"Fraudes: {df['Class'].mean()*100:.4f}%\")

In [ ]:
ax = df['Class'].value_counts().sort_index().plot(kind='bar', title='Distribuição das classes')\nax.set_xticklabels(['Legítima', 'Fraude'], rotation=0)\nax.set_ylabel('Quantidade')\nplt.show()

**Decisão:** como a classe positiva é extremamente rara, acurácia não será a métrica principal. O foco é Recall da fraude, acompanhado de Precision, F1 e AUC-PR.

## 2. Preparação dos dados

### Engenharia de atributos\nAlém da padronização, criamos `Amount_log = log1p(Amount)`. A transformação logarítmica reduz a assimetria de valores monetários muito altos sem perder transações de valor zero. Mantemos `Amount` para que os modelos também possam aprender com o valor original.

In [ ]:
df_model = df.copy()
df_model['Amount_log'] = np.log1p(df_model['Amount'])
X = df_model.drop(columns='Class')
y = df_model['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
preprocessor = ColumnTransformer([('scale', StandardScaler(), ['Time','Amount','Amount_log'])], remainder='passthrough')
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print('Treino:', X_train.shape, '| Teste:', X_test.shape)
print('Fraudes treino/teste:', int(y_train.sum()), int(y_test.sum()))

## 3. Treinamento e comparação

In [ ]:
models = {\n 'Regressão Logística': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),\n 'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),\n 'XGBoost': XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, eval_metric='logloss', n_jobs=-1, random_state=RANDOM_STATE)\n}\npipelines, probabilities, rows = {}, {}, []\nfor name, model in models.items():\n    pipe = Pipeline([('prep', preprocessor), ('model', model)])\n    pipe.fit(X_train, y_train)\n    prob = pipe.predict_proba(X_test)[:,1]\n    pred = (prob >= 0.5).astype(int)\n    pipelines[name], probabilities[name] = pipe, prob\n    rows.append({'Modelo':name, 'Precision fraude':precision_score(y_test,pred,zero_division=0), 'Recall fraude':recall_score(y_test,pred), 'F1 fraude':f1_score(y_test,pred), 'ROC-AUC':roc_auc_score(y_test,prob), 'AUC-PR':average_precision_score(y_test,prob)})\nresults = pd.DataFrame(rows).sort_values('Recall fraude', ascending=False).reset_index(drop=True)\ndisplay(results)

In [ ]:
for name in results['Modelo']:\n    pred = (probabilities[name] >= 0.5).astype(int)\n    print('\\n', '='*60, '\\n', name)\n    print(classification_report(y_test, pred, target_names=['Legítima','Fraude'], digits=4))\n    sns.heatmap(confusion_matrix(y_test,pred), annot=True, fmt='d')\n    plt.title('Matriz de confusão — ' + name); plt.xlabel('Previsto'); plt.ylabel('Real'); plt.show()

### Curvas ROC e Precision-Recall\nA ROC mostra a relação entre taxa de verdadeiros positivos e falsos positivos. Em bases muito desbalanceadas, a curva Precision-Recall é especialmente útil porque evidencia diretamente o equilíbrio entre encontrar fraudes e gerar falsos alertas.

In [ ]:
from sklearn.metrics import roc_curve
plt.figure(figsize=(8,5))
for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],linestyle='--')
plt.xlabel('Taxa de falsos positivos'); plt.ylabel('Recall / TPR')
plt.title('Curvas ROC'); plt.legend(); plt.show()

plt.figure(figsize=(8,5))
for name, prob in probabilities.items():
    p, r, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    plt.plot(r, p, label=f'{name} (AP={ap:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Curvas Precision-Recall'); plt.legend(); plt.show()

## 4. Ajuste do threshold\nPara fins didáticos, o modelo com maior Recall em 0,50 é usado para demonstrar a busca de um limiar com Recall ≥ 90%, escolhendo entre os candidatos aquele com maior Precision. Em produção, o threshold deve ser escolhido em validação separada.

In [ ]:
best_name = results.iloc[0]['Modelo']\nbest_prob = probabilities[best_name]\nprecision, recall, thresholds = precision_recall_curve(y_test, best_prob)\ntab = pd.DataFrame({'threshold':thresholds, 'precision':precision[:-1], 'recall':recall[:-1]})\ncandidates = tab[tab['recall'] >= 0.90]\nchosen = candidates.sort_values(['precision','threshold'], ascending=[False,False]).iloc[0] if len(candidates) else tab.iloc[(tab['recall']-0.90).abs().argmin()]\nbest_threshold = float(chosen['threshold'])\npred_adj = (best_prob >= best_threshold).astype(int)\nprint('Modelo:', best_name, '| Threshold:', round(best_threshold,6))\nprint('Precision:', precision_score(y_test,pred_adj,zero_division=0))\nprint('Recall:', recall_score(y_test,pred_adj))\nprint('F1:', f1_score(y_test,pred_adj))\nprint(confusion_matrix(y_test,pred_adj))\nplt.plot(tab['threshold'],tab['precision'],label='Precision'); plt.plot(tab['threshold'],tab['recall'],label='Recall'); plt.axvline(best_threshold,linestyle='--',label='Threshold escolhido'); plt.xlabel('Threshold'); plt.ylabel('Score'); plt.legend(); plt.show()

## 5. Explicabilidade com SHAP

In [ ]:
tree_results = results[results['Modelo'].isin(['Random Forest','XGBoost'])]\ntree_name = tree_results.iloc[0]['Modelo']\npipe = pipelines[tree_name]\nsample = X_test.sample(min(1000,len(X_test)), random_state=RANDOM_STATE)\nXt = pipe.named_steps['prep'].transform(sample)\nnames = pipe.named_steps['prep'].get_feature_names_out()\nexplainer = shap.TreeExplainer(pipe.named_steps['model'])\nsv = explainer.shap_values(Xt)\nif isinstance(sv,list): sv = sv[-1]\nelif np.ndim(sv)==3: sv = sv[:,:,-1]\nprint('Modelo explicado:', tree_name)\nshap.summary_plot(sv, Xt, feature_names=names, show=True)

### Importância das variáveis\nAlém do SHAP, mostramos a importância global calculada pelo próprio modelo de árvores. Isso oferece uma segunda visão sobre quais componentes mais influenciam a classificação.

In [ ]:
if hasattr(pipe.named_steps['model'], 'feature_importances_'):
    imp = pd.Series(pipe.named_steps['model'].feature_importances_, index=names).sort_values(ascending=False).head(15)
    imp.sort_values().plot(kind='barh', title=f'Top 15 variáveis — {tree_name}')
    plt.xlabel('Importância'); plt.show()

## Conclusão\nO projeto evidencia que a escolha da métrica é fundamental em dados desbalanceados. Recall mede a capacidade de encontrar fraudes reais; Precision ajuda a controlar falsos alertas. O threshold permite ajustar esse equilíbrio, enquanto SHAP auxilia na interpretação do modelo. Os resultados numéricos devem ser lidos após a execução do notebook e não foram inventados previamente.